In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn import tree
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [2]:
data = pd.read_csv("C:/Users/N/Videos/attacks_clone/tree-adversarial-attacks/data/numerical.csv")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
features = ['LdrGetProcedureAddress', 'GetSystemInfo', 'GetNativeSystemInfo', 'NtQuerySystemInformation',  'CreateToolhelp32Snapshot', 'Process32FirstW', 'RegOpenKeyExW', 'RegQueryValueExW',
            'RegCloseKey', 'NtCreateFile', 'GetDiskFreeSpaceW', 'CryptAcquireContextW', 'CryptGenKey', 
            'CryptExportKey', 'RegCreateKeyExW', 'RegSetValueExW', 'CreateThread', 'FindFirstFileExW',
            'NtWriteFile', 'ShellExecuteExW', 'InternetOpenW', 
            'InternetConnectW', 'HttpOpenRequestW', 'InternetReadFile', 'InternetOpenUrlA', 
            'InternetCloseHandle']


benign_features = ['LdrGetDllHandle',
 'RegOpenKeyExW',
 'RegQueryValueExW',
 'RegCloseKey',
 'GetSystemWindowsDirectoryW',
 'CreateActCtxW',
 'GetSystemDirectoryW',
 'GetVolumeNameForVolumeMountPointW',
 'RegEnumKeyW',
 'GlobalMemoryStatusEx',
 'CreateDirectoryW',
 'GetVolumePathNamesForVolumeNameW',
 'LdrUnloadDll',
 'GetFileAttributesW',
 'SHGetFolderPathW',
 'GetSystemMetrics',
 'SetFileTime',
 'RegEnumKeyExW',
 'DrawTextExW',
 'FindWindowW',
 'FindWindowExA',
 'RemoveDirectoryW',
 'GetKeyboardState',
 'SendNotifyMessageW',
 'GetFileVersionInfoSizeW',
 'EnumWindows',
 'GetUserNameExA',
 'RegCreateKeyExA',
 'SetInformationJobObject',
 'DrawTextExA',
 'RegSetValueExA',
 'SHGetSpecialFolderLocation',
 'gethostbyname',
 'CertOpenStore',
 'CertControlStore',
 'SetWindowsHookExW',
 'WSASocketW',
 'WSAConnect',
 'UnhookWindowsHookEx',
 'CertOpenSystemStoreW',
 'HttpQueryInfoA',
 'CryptProtectMemory',
 'WriteProcessMemory',
 'SendNotifyMessageA',
 'JsGlobalObjectDefaultEvalHelper',
 'ObtainUserAgentString',
 'label'
 ]

In [4]:
benign_filtered = [item for item in benign_features if item not in features]
functionality_features = features.extend(benign_filtered)

In [5]:
print(len(features))

70


In [6]:
df = data[features]

In [7]:
df.shape

(3083, 70)

In [8]:
df.head()

,LdrGetProcedureAddress,GetSystemInfo,GetNativeSystemInfo,NtQuerySystemInformation,CreateToolhelp32Snapshot,Process32FirstW,RegOpenKeyExW,RegQueryValueExW,RegCloseKey,NtCreateFile,GetDiskFreeSpaceW,CryptAcquireContextW,CryptGenKey,CryptExportKey,RegCreateKeyExW,RegSetValueExW,CreateThread,FindFirstFileExW,NtWriteFile,ShellExecuteExW,InternetOpenW,InternetConnectW,HttpOpenRequestW,InternetReadFile,InternetOpenUrlA,InternetCloseHandle,LdrGetDllHandle,GetSystemWindowsDirectoryW,CreateActCtxW,GetSystemDirectoryW,GetVolumeNameForVolumeMountPointW,RegEnumKeyW,GlobalMemoryStatusEx,CreateDirectoryW,GetVolumePathNamesForVolumeNameW,LdrUnloadDll,GetFileAttributesW,SHGetFolderPathW,GetSystemMetrics,SetFileTime,RegEnumKeyExW,DrawTextExW,FindWindowW,FindWindowExA,RemoveDirectoryW,GetKeyboardState,SendNotifyMessageW,GetFileVersionInfoSizeW,EnumWindows,GetUserNameExA,RegCreateKeyExA,SetInformationJobObject,DrawTextExA,RegSetValueExA,SHGetSpecialFolderLocation,gethostbyname,CertOpenStore,CertControlStore,SetWindowsHookExW,WSASocketW,WSAConnect,UnhookWindowsHookEx,CertOpenSystemStoreW,HttpQueryInfoA,CryptProtectMemory,WriteProcessMemory,SendNotifyMessageA,JsGlobalObjectDefaultEvalHelper,ObtainUserAgentString,label
0,1426,10,9,7,0,0,1547,1308,7834,7398,0,1,0,0,235,271,6,17478,40704,5,0,0,0,2,2,7,141,58,6,25,21,8,1,9,48,74,149,8,670,3317,128,86,0,0,0,0,0,9,6,0,4,0,0,4,6,0,0,0,0,0,0,0,0,0,4,3,0,0,0,1
1,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,564,162,1,4,0,0,195,317,136,1213,0,213,0,0,2,1,7,0,351,1,0,0,0,0,0,0,200,10,5,11,3,0,13,3,8,7,25,11,6,0,38,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,245,92,0,2,0,0,106,182,66,95,0,0,0,0,1,0,4,0,2244,0,0,0,0,0,0,0,70,2,3,4,0,0,13,4,0,2,11,2,1,0,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,0,0,0,0,0,0,0,0,1
4,5,2,0,0,0,0,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [10]:
scale = StandardScaler()
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
X_scaled = scale.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [11]:
clf = RandomForestClassifier()
clf = clf.fit(X_train,y_train)
y_pred_clf = clf.predict(X_test)
accuracy_clf = accuracy_score(y_test, y_pred_clf)
print(f'Accuracy: {accuracy_clf * 100:.2f}%')

Accuracy: 98.06%


In [12]:
print(X_train.shape)

(2466, 69)


In [13]:
poisoned_samples = np.load("C:/Users/N/Videos/attacks_clone/tree-adversarial-attacks/adversarial_samples/adversarial_samples_DT/DT_adversarial_samples_functionality_preserving_2.npy")

In [14]:
len(poisoned_samples)

357

In [15]:
print(poisoned_samples)

[[  0   0   0 ...   0   0   0]
 [210   3   2 ...   0   0   0]
 [  0   0   0 ...   0   0   0]
 ...
 [  5   2   0 ...   0   0   0]
 [ 56   0   0 ...   0   0   0]
 [  0   0   0 ...   0   0   0]]


In [16]:
poisoned_samples_scaled = scale.fit_transform(poisoned_samples)

In [17]:
poisoned_samples_scaled

array([[-0.21655457, -0.07396574, -0.40568701, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.65536252, -0.05782529,  2.84892562, ...,  0.        ,
         0.        ,  0.        ],
       [-0.21655457, -0.07396574, -0.40568701, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.19579464, -0.06320544, -0.40568701, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.01595665, -0.07396574, -0.40568701, ...,  0.        ,
         0.        ,  0.        ],
       [-0.21655457, -0.07396574, -0.40568701, ...,  0.        ,
         0.        ,  0.        ]], shape=(357, 69))

In [18]:
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = clf.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"adversarial Samples successful: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(poisoned_samples_scaled)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [19]:
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred_xgb = model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'Accuracy: {accuracy_xgb * 100:.2f}%')

Accuracy: 97.89%


In [20]:
def make_pred_poisoned_XGB(samples):
    predictions = [] 
    for sample in samples:
        single_pred = model.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned_XGB(poisoned_samples_scaled)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [21]:
svm_classifier = SVC(kernel='linear', C=1.0, random_state=42)
svm_classifier.fit(X_train, y_train)
y_pred = svm_classifier.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

Accuracy: 0.97


In [22]:
def make_pred_poisoned_SVM(samples):
    predictions = [] 
    for sample in samples:
        single_pred = svm_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("sample adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned_SVM(poisoned_samples_scaled)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[0]
sample adversarial
[1]
Correctly identified adversarial ransomware sample
[0]
sample adversarial
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[0]
sample adversarial
[0]
sample adversarial
[0]
sample adversarial
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[0]
sample adversarial
[0]
sample adversarial
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[0]
sample adversarial
[0]
sample adversarial
[1]
Correctly identifie

In [26]:
LR_model = LogisticRegression(solver='saga',max_iter=4000)
LR_model.fit(X_train, y_train)
accuracy = accuracy_score(y_test, y_pred)
print(f"accuracy: {round(accuracy,2)}")

accuracy: 0.97


In [29]:
def make_pred_poisoned_LR(samples):
    predictions = [] 
    for sample in samples:
        single_pred = LR_model.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned_LR(poisoned_samples_scaled)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc